# Get Yahoo Public Picks

Used to compare my picks with general public concensus. Assumes yahoo data (https://tournament.fantasysports.yahoo.com/mens-basketball-bracket/pickdistribution?year=2024) has been exported to Excel. 

Remember that you may have to manually update play-ins in the code.

In [1]:
season = 2025

play_in_updates = [
    # (yahoo_name, corrected_name) for every play-in matchup
    ('ALST/SFPA', 'ALST'),
    ('AMER/MSM', 'AMER'),
    ('SDSU/UNC', 'UNC'),
    ('TEX/XAV', 'XAV'),
]

season, play_in_updates

(2025,
 [('ALST/SFPA', 'ALST'),
  ('AMER/MSM', 'AMER'),
  ('SDSU/UNC', 'UNC'),
  ('TEX/XAV', 'XAV')])

In [2]:
import pandas as pd

pd.set_option('display.max_columns', 100)

df = pd.concat(
    [
        pd.read_excel(fr'..\data\unprocessed\mens_yahoo\yahoo_picks_{season}.xlsx', sheet_name=f'round_{round_}')
        .assign(Round=round_)
        for round_ in range(1, 7)
    ],
    ignore_index=True,
)

df['Seed'] = df['Team (Seed)'].str.extract(r'\((\d+)\)').astype(int)
df['Team (Seed)'] = df['Team (Seed)'].str.replace(r'\(\d+\)', '', regex=True)

df.rename(columns={'Team (Seed)': 'Team'}, inplace=True)

df = df.pivot(index=['Team', 'Seed'], columns=['Round'], values='% Picked').reset_index()
df.columns = ['Team', 'Seed', 'Round 1', 'Round 2', 'Round 3', 'Round 4', 'Round 5', 'Round 6']

df

,Team,Seed,Round 1,Round 2,Round 3,Round 4,Round 5,Round 6
0,ALST/SFPA,16,0.0162,0.0070,0.0038,0.0021,0.0010,0.0005
1,AMER/MSM,16,0.0163,0.0075,0.0038,0.0022,0.0010,0.0004
2,Akron,13,0.1202,0.0423,0.0062,0.0024,0.0011,0.0006
3,Alabama,2,0.9460,0.7902,0.5595,0.1790,0.0848,0.0332
4,Arizona,4,0.8519,0.5201,0.0898,0.0394,0.0162,0.0075
...,...,...,...,...,...,...,...,...
59,VCU,11,0.3143,0.0792,0.0153,0.0031,0.0014,0.0006
60,Vanderbilt,10,0.4162,0.0552,0.0199,0.0042,0.0016,0.0007
61,Wisconsin,3,0.9184,0.6027,0.2287,0.0583,0.0248,0.0105
62,Wofford,15,0.0336,0.0128,0.0064,0.0024,0.0012,0.0005


Fix play-ins

In [3]:
df.loc[df['Team'].str.contains('/', regex=False), :]

,Team,Seed,Round 1,Round 2,Round 3,Round 4,Round 5,Round 6
0,ALST/SFPA,16,0.0162,0.0070,0.0038,0.0021,0.0010,0.0005
1,AMER/MSM,16,0.0163,0.0075,0.0038,0.0022,0.0010,0.0004
47,SDSU/UNC,11,0.3195,0.1023,0.0203,0.0065,0.0033,0.0022
51,TEX/XAV,11,0.2246,0.0571,0.0107,0.0032,0.0013,0.0007


In [4]:
for yahoo_name, corrected_name in play_in_updates:
    df.loc[df['Team'] == yahoo_name, 'Team'] = corrected_name

df.loc[df['Team'].str.contains('/', regex=False), :]

,Team,Seed,Round 1,Round 2,Round 3,Round 4,Round 5,Round 6


Map with Kaggle data

In [5]:
df_seeds = pd.read_csv(r'..\data\unprocessed\kaggle\MNCAATourneySeeds.csv')

df_seeds = df_seeds.loc[df_seeds['Season'] == season, :].reset_index(drop=True)

df_seeds.insert(2, 'Play In', df_seeds['Seed'].str.endswith(('a', 'b')))
df_seeds.insert(2, 'Region', df_seeds['Seed'].str[0])
df_seeds['Seed'] = df_seeds['Seed'].str.extract('(\d+)').astype(int)

df_seeds

,Season,Seed,Region,Play In,TeamID
0,2025,1,W,False,1181
1,2025,2,W,False,1104
2,2025,3,W,False,1458
3,2025,4,W,False,1112
4,2025,5,W,False,1332
...,...,...,...,...,...
63,2025,12,Z,False,1161
64,2025,13,Z,False,1213
65,2025,14,Z,False,1423
66,2025,15,Z,False,1303


In [6]:
df_spellings = pd.read_csv(
    r'..\data\unprocessed\kaggle\MTeamSpellings.csv', 
    encoding='cp1252'  # fixes issue with fancy quotes
)

df_spellings.loc[df_spellings.shape[0]] = ['fdu', 1192]
df_spellings.loc[df_spellings.shape[0]] = ['sdsu', 1361]
df_spellings.loc[df_spellings.shape[0]] = ['csu', 1161]

df_spellings

,TeamNameSpelling,TeamID
0,a&m-corpus chris,1394
1,a&m-corpus christi,1394
2,abilene chr,1101
3,abilene christian,1101
4,abilene-christian,1101
...,...,...
1175,youngstown-st,1464
1176,youngstown-state,1464
1177,fdu,1192
1178,sdsu,1361


In [7]:
df_spellings = pd.merge(
    df_spellings,
    df_seeds[['TeamID', 'Seed']],
    how='inner',
    on=['TeamID']
)

df_spellings

,TeamNameSpelling,TeamID,Seed
0,akron,1103,13
1,alabama,1104,2
2,alabama st,1106,16
3,alabama st.,1106,16
4,alabama state,1106,16
...,...,...,...
171,wofford,1459,15
172,xavier,1462,11
173,yale,1463,13
174,sdsu,1361,11


In [8]:
from fuzzywuzzy.fuzz import token_sort_ratio
from fuzzywuzzy import process
from tqdm.autonotebook import tqdm

team_spellings = df_spellings['TeamNameSpelling'].unique()
yahoo_teams = df.loc[~df['Team'].str.contains('^playin', regex=True), 'Team'].unique()

df_match = pd.DataFrame(
    [
        [
            yahoo_team,
            *process.extract(
                yahoo_team,
                team_spellings,
                scorer=token_sort_ratio,
                limit=1
            )[0][:2]
        ] for yahoo_team in tqdm(yahoo_teams)
    ],
    columns=['Yahoo Team', 'Team Spelling', 'Match Score']
).sort_values('Match Score', ignore_index=True)

df_match.head(25)

C:\Users\mhugh\AppData\Local\Temp\ipykernel_22580\794795546.py:3: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


  0%|          | 0/64 [00:00<?, ?it/s]

,Yahoo Team,Team Spelling,Match Score
0,ALST,alabama st,57
1,AMER,american,67
2,XAV,xavier,67
3,N.C. Wilmington,nc wilmington,89
4,Michigan St.,michigan st,100
5,Mississippi,mississippi,100
6,Mississippi St.,mississippi st,100
7,Missouri,missouri,100
8,Montana,montana,100
9,Nebraska Omaha,nebraska omaha,100


In [9]:
yahoo_to_spelling = dict(zip(df_match['Yahoo Team'], df_match['Team Spelling']))
spelling_to_id = dict(zip(df_spellings['TeamNameSpelling'], df_spellings['TeamID']))

df.insert(1, 'TeamID', df['Team'].map(yahoo_to_spelling).map(spelling_to_id))

df

,Team,TeamID,Seed,Round 1,Round 2,Round 3,Round 4,Round 5,Round 6
0,ALST,1106,16,0.0162,0.0070,0.0038,0.0021,0.0010,0.0005
1,AMER,1110,16,0.0163,0.0075,0.0038,0.0022,0.0010,0.0004
2,Akron,1103,13,0.1202,0.0423,0.0062,0.0024,0.0011,0.0006
3,Alabama,1104,2,0.9460,0.7902,0.5595,0.1790,0.0848,0.0332
4,Arizona,1112,4,0.8519,0.5201,0.0898,0.0394,0.0162,0.0075
...,...,...,...,...,...,...,...,...,...
59,VCU,1433,11,0.3143,0.0792,0.0153,0.0031,0.0014,0.0006
60,Vanderbilt,1435,10,0.4162,0.0552,0.0199,0.0042,0.0016,0.0007
61,Wisconsin,1458,3,0.9184,0.6027,0.2287,0.0583,0.0248,0.0105
62,Wofford,1459,15,0.0336,0.0128,0.0064,0.0024,0.0012,0.0005


In [10]:
import numpy as np

id_to_playin = dict(zip(df_seeds['TeamID'], df_seeds['Play In']))
id_to_playin[np.nan] = True

df.insert(3, 'Play In', df['TeamID'].map(id_to_playin))

df

,Team,TeamID,Seed,Play In,Round 1,Round 2,Round 3,Round 4,Round 5,Round 6
0,ALST,1106,16,True,0.0162,0.0070,0.0038,0.0021,0.0010,0.0005
1,AMER,1110,16,True,0.0163,0.0075,0.0038,0.0022,0.0010,0.0004
2,Akron,1103,13,False,0.1202,0.0423,0.0062,0.0024,0.0011,0.0006
3,Alabama,1104,2,False,0.9460,0.7902,0.5595,0.1790,0.0848,0.0332
4,Arizona,1112,4,False,0.8519,0.5201,0.0898,0.0394,0.0162,0.0075
...,...,...,...,...,...,...,...,...,...,...
59,VCU,1433,11,False,0.3143,0.0792,0.0153,0.0031,0.0014,0.0006
60,Vanderbilt,1435,10,False,0.4162,0.0552,0.0199,0.0042,0.0016,0.0007
61,Wisconsin,1458,3,False,0.9184,0.6027,0.2287,0.0583,0.0248,0.0105
62,Wofford,1459,15,False,0.0336,0.0128,0.0064,0.0024,0.0012,0.0005


In [11]:
id_to_region = dict(zip(df_seeds['TeamID'], df_seeds['Region']))

df.insert(3, 'Region', df['TeamID'].map(id_to_region))

df

,Team,TeamID,Seed,Region,Play In,Round 1,Round 2,Round 3,Round 4,Round 5,Round 6
0,ALST,1106,16,Y,True,0.0162,0.0070,0.0038,0.0021,0.0010,0.0005
1,AMER,1110,16,W,True,0.0163,0.0075,0.0038,0.0022,0.0010,0.0004
2,Akron,1103,13,W,False,0.1202,0.0423,0.0062,0.0024,0.0011,0.0006
3,Alabama,1104,2,W,False,0.9460,0.7902,0.5595,0.1790,0.0848,0.0332
4,Arizona,1112,4,W,False,0.8519,0.5201,0.0898,0.0394,0.0162,0.0075
...,...,...,...,...,...,...,...,...,...,...,...
59,VCU,1433,11,W,False,0.3143,0.0792,0.0153,0.0031,0.0014,0.0006
60,Vanderbilt,1435,10,W,False,0.4162,0.0552,0.0199,0.0042,0.0016,0.0007
61,Wisconsin,1458,3,W,False,0.9184,0.6027,0.2287,0.0583,0.0248,0.0105
62,Wofford,1459,15,X,False,0.0336,0.0128,0.0064,0.0024,0.0012,0.0005


In [12]:
df.loc[df['Region'].isna(), :]

,Team,TeamID,Seed,Region,Play In,Round 1,Round 2,Round 3,Round 4,Round 5,Round 6


Redistribute play-in probabilities to the teams that won

If using 2023, it is unclear which play-in teams are which

In [13]:
# for seed in df.loc[df['Play In'], 'Seed'].unique():
#     df.loc[
#         (~df['Team'].str.contains('^playin', regex=True)) & 
#         (df['Seed'] == seed) &
#         (df['Play In']), 
#         [f'Round {i}' for i in range(1, 7)]
#     ] += df.loc[
#         (df['Team'].str.contains('^playin', regex=True)) & 
#         (df['Seed'] == seed) &
#         (df['Play In']), 
#         [f'Round {i}' for i in range(1, 7)]
#     ].mean(axis=0)

# df = df.loc[
#     ~df['Team'].str.contains('^playin', regex=True), 
#     :
# ].reset_index(drop=True)

# df['TeamID'] = df['TeamID'].astype(int)

# df.loc[df['Play In'], :]

Redistribute percentages to account for rounding inaccuracies

In [14]:
# for i in range(1, 7):
#     df[f'Round {i}'] = df[f'Round {i}'] / df[f'Round {i}'].sum() * 2**(6 - i)

# df

In [15]:
df.sum()

Team       ALSTAMERAkronAlabamaArizonaArkansasAuburnBYUBa...
TeamID                                                 81523
Seed                                                     544
Region     YWWWWZYWWYXZZYZWZXXZXXXYZXWYYYZXZYYYWZWZZYZZWX...
Play In                                                    4
Round 1                                              31.2225
Round 2                                              15.7073
Round 3                                               7.8877
Round 4                                               3.9548
Round 5                                               1.9977
Round 6                                               0.9998
dtype: object

In [16]:
df.insert(df.columns.get_loc('Seed'), 'Region Seed', df['Region'] + df['Seed'].astype(str).str.zfill(2))

df

,Team,TeamID,Region Seed,Seed,Region,Play In,Round 1,Round 2,Round 3,Round 4,Round 5,Round 6
0,ALST,1106,Y16,16,Y,True,0.0162,0.0070,0.0038,0.0021,0.0010,0.0005
1,AMER,1110,W16,16,W,True,0.0163,0.0075,0.0038,0.0022,0.0010,0.0004
2,Akron,1103,W13,13,W,False,0.1202,0.0423,0.0062,0.0024,0.0011,0.0006
3,Alabama,1104,W02,2,W,False,0.9460,0.7902,0.5595,0.1790,0.0848,0.0332
4,Arizona,1112,W04,4,W,False,0.8519,0.5201,0.0898,0.0394,0.0162,0.0075
...,...,...,...,...,...,...,...,...,...,...,...,...
59,VCU,1433,W11,11,W,False,0.3143,0.0792,0.0153,0.0031,0.0014,0.0006
60,Vanderbilt,1435,W10,10,W,False,0.4162,0.0552,0.0199,0.0042,0.0016,0.0007
61,Wisconsin,1458,W03,3,W,False,0.9184,0.6027,0.2287,0.0583,0.0248,0.0105
62,Wofford,1459,X15,15,X,False,0.0336,0.0128,0.0064,0.0024,0.0012,0.0005


In [17]:
df.to_parquet(f'../data/preprocessed/mens_yahoo/yahoo_picks_{season}.parquet')

'Done'

'Done'